In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from Bio import SeqIO
import os
import statistics
import shutil
import seaborn as sns
import re

In [ ]:
pwd = os.getcwd().split('/')[-1]

match = re.search(r'(\d+)$', pwd)

if not match:
    print("Error: No numeric suffix found in the directory name")
    exit(1)

suffix = int(match.group(1)) - 1
print(f"Suffix: {suffix}")

first = False
if suffix == 0:
    first = True

species_list = ["Osa","Ogl","Ola","Lpe","Hvu","Ata","Tae","Bdi","Svi","Pvi","Pha","Msi","Sbi","Zma","Ecu","Pau"]

In [ ]:
if first:
    import orthofinder_to_fasta

    orthogroups_file = "N0.tsv"
    fasta_dir = "/pscratch/sd/m/mashiana/grassfam2025/orthofinder/grass"
    output_dir = "hogDirectory"

    orthogroups = orthofinder_to_fasta.parse_orthogroups(orthogroups_file)

    orthofinder_to_fasta.extract_sequences(orthogroups, fasta_dir, output_dir)

    print(f"FASTA files created in {output_dir}")

In [ ]:
if not first:
    old_path = '/'.join(os.getcwd().split('/')[:-1]) + '/' + re.sub(r'\d+$', str(suffix), pwd)

    super_matrix = pd.read_csv(os.path.join(old_path, 'grassHmmDatabase/grassfam1/superMatrix'), delimiter='\t')
    hog_hmm_disagree = super_matrix[super_matrix["Hog"] != super_matrix["HmmHogHit"]]
    gene_and_hog_to_remove = hog_hmm_disagree.groupby(["Hog", "HmmHogHit"])["Gene"].apply(list).to_dict()

    hogList = pd.read_csv(os.path.join(old_path, 'grassHmmDatabase/grassfam1/geneInfo'), delimiter='\t')['gene']
    geneInfo = pd.read_csv(os.path.join(old_path, 'grassHmmDatabase/grassfam1/geneInfo'), delimiter='\t')
    hmmList = pd.read_csv(os.path.join(old_path, 'grassHmmDatabase/grassfam1/filtered_confusion_df'), delimiter='\t')['Gene']

    hogSet = set(hogList)
    hmmSet = set(hmmList)

    unique_to_hog = hogSet - hmmSet

    filtered_geneInfo = geneInfo[geneInfo['gene'].isin(unique_to_hog)]
    zeros_to_remove = filtered_geneInfo.groupby("hog")["gene"].apply(list).to_dict()
    
    shutil.copytree(f"{old_path}/hogDirectoryFiltered", "./hogDirectory")

In [ ]:
if not first:
    directory = './hogDirectory'

    for filename in os.listdir(directory):
        if filename.endswith(".fasta"):
            filepath = os.path.join(directory, filename)
            
            sequences = {}
            with open(filepath, "r") as fasta_file:
                for record in SeqIO.parse(fasta_file, "fasta"):
                    sequences[record.id] = str(record.seq)
            
            hog_id = filename.split(".")[1]

            # Remove all genes that were nohits in this fasta
            genes_to_keep2 = zeros_to_remove.get(hog_id, [])
            for gene in genes_to_keep2:
                sequences.pop(gene, None)

            # for every unique hog_key2 where the hog_key1 matches hog_id, make a new file with the genes retunred by sequences.pop(gene, None)
            file_counter = 0
            for (hog_key1, hog_key2), genes in gene_and_hog_to_remove.items():
                if hog_id == hog_key1:
                    file_counter += 1
                    removed_sequences = {}
                    for gene in genes:
                        removed_sequences[gene] = sequences.pop(gene)

                    # Write removed sequences to a new file with -file_counter appended
                    new_file_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}-{file_counter}.fasta")
                    with open(new_file_path, "w") as new_fasta_file:
                        for gene_id, sequence in removed_sequences.items():
                            new_fasta_file.write(f">{gene_id}\n{sequence}\n")


            if (file_counter == 0):
                # Write the origional name since no subsets were made
                with open(filepath, "w") as fasta_file:
                    for gene_id, sequence in sequences.items():
                        fasta_file.write(f">{gene_id}\n{sequence}\n")
            else:
                # Write the origional wiht -0 appended
                original_file_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}-0.fasta")
                with open(original_file_path, "w") as fasta_file:
                    for gene_id, sequence in sequences.items():
                        fasta_file.write(f">{gene_id}\n{sequence}\n")
                # remove non -0 appended origional
                os.remove(filepath)


    # empty -0 files were with ratio = 1.0, so remove the -0 file and change it back to no suffix since they will be persistant through this process


    for filename in os.listdir(directory):
        if filename.endswith("-0.fasta"):
            filepath = os.path.join(directory, filename)
            hog_id = filename.split(".")[1].split("-")[0]

            if os.path.getsize(filepath) == 0:
                print(filename)
                os.remove(filepath)

                for related_file in os.listdir(directory):
                    related_hog_id = related_file.split(".")[1].split("-")[0]
                    if related_hog_id == hog_id and related_file.endswith("-1.fasta"):
                        new_filename = related_file.replace("-1.fasta", ".fasta")
                        os.rename(os.path.join(directory, related_file), os.path.join(directory, new_filename))

    # remove all empty files, this is an edge case where all genes split off, but not significantly to their new split, 
    # so they actually leave the old one empty, but it doesn't get the -0 suffix since no derivations are made

    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if os.path.getsize(filepath) == 0:
            print(f"Removing empty file: {filename}")
            os.remove(filepath)

Get new Hog Seq info

In [ ]:
output_tsv = './hogMsaStats.tsv'
hog_directory = './hogDirectory'

with open(output_tsv, 'w') as f_out:
    # Write the header row
    species_columns = "\t".join(species_list)
    f_out.write(f"HOG\tmin\tmax\tmedian\tmean\tsd\tspecies_count\tgene_count\tlistOfLengths\t{species_columns}\n")

    # Iterate through files in the MSA directory
    for filename in os.listdir(hog_directory):
        if filename.endswith(".fasta"):
            hog_id = filename.split('.')[1]
            hog_file = os.path.join(hog_directory, filename)

            species = set()
            lengths = []
            for record in SeqIO.parse(hog_file, "fasta"):
                lengths.append(len(record.seq))
                specie = record.id.split('|')[0]
                species.add(specie)

            gene_count = len(lengths)
            min_length = min(lengths)
            max_length = max(lengths)
            median_length = statistics.median(lengths)
            mean_length = statistics.mean(lengths)
            if gene_count < 2:
                sd_length = -1
            else:
                sd_length = statistics.stdev(lengths)
            lengths_str = ','.join(map(str, lengths))
            species_count = len(species)

            species_binary = [str(1 if specie in species else 0) for specie in species_list]

            # Write the statistics to the TSV file
            species_binary_str = "\t".join(species_binary)
            f_out.write(f"{hog_id}\t{min_length}\t{max_length}\t{median_length}\t{mean_length}\t{sd_length}\t{species_count}\t{gene_count}\t{lengths_str}\t{species_binary_str}\n")


In [ ]:
hog_directory="./hogDirectory"
destination_directory="./hogDirectoryFiltered"
hog_stats="./hogMsaStats.tsv"

pacmad = ['Svi', 'Pvi', 'Pha', 'Msi', 'Sbi', 'Zma', 'Ecu', 'Pau']
bop = ['Osa', 'Ogl', 'Ola', 'Lpe', 'Hvu', 'Ata', 'Tae', 'Bdi']

os.makedirs(destination_directory, exist_ok=True)

hog_stats = pd.read_csv(hog_stats, sep='\t')

get_hog_file = lambda hog: os.path.join(hog_directory, f"N0.{hog}.fasta")

def filter_hog(hog):
    # Calculate species count inside the function
    species_count = hog['species_count']

    found_pacmad = sum(int(hog[species]) == 1 for species in pacmad)
    found_bop = sum(int(hog[species]) == 1 for species in bop)

    # Check condition: species count total over 8 either both have >0, or one is 0 and the other >= 6
    if (species_count >= 8) or \
       (found_pacmad == 0 and found_bop >= 6) or \
       (found_bop == 0 and found_pacmad >= 6):
        
        hog_file = get_hog_file(hog['HOG'])
        if os.path.exists(hog_file):
            shutil.copy(hog_file, destination_directory)
        else:
            print(f"File not found: {hog_file}")

# Apply to all rows, no pre-filtering here
hog_stats.apply(filter_hog, axis=1)

In [ ]:
output_tsv = './hogMsaStats2.tsv'
hog_directory = './hogDirectoryFiltered'

with open(output_tsv, 'w') as f_out:
    # Write the header row
    species_columns = "\t".join(species_list)
    f_out.write(f"HOG\tmin\tmax\tmedian\tmean\tsd\tspecies_count\tgene_count\tlistOfLengths\t{species_columns}\n")

    # Iterate through files in the MSA directory
    for filename in os.listdir(hog_directory):
        if filename.endswith(".fasta"):
            hog_id = filename.split('.')[1]
            hog_file = os.path.join(hog_directory, filename)

            species = set()
            lengths = []
            for record in SeqIO.parse(hog_file, "fasta"):
                lengths.append(len(record.seq))
                specie = record.id.split('|')[0]
                species.add(specie)

            gene_count = len(lengths)
            min_length = min(lengths)
            max_length = max(lengths)
            median_length = statistics.median(lengths)
            mean_length = statistics.mean(lengths)
            if gene_count < 2:
                sd_length = -1
            else:
                sd_length = statistics.stdev(lengths)
            lengths_str = ','.join(map(str, lengths))
            species_count = len(species)

            species_binary = [str(1 if specie in species else 0) for specie in species_list]

            # Write the statistics to the TSV file
            species_binary_str = "\t".join(species_binary)
            f_out.write(f"{hog_id}\t{min_length}\t{max_length}\t{median_length}\t{mean_length}\t{sd_length}\t{species_count}\t{gene_count}\t{lengths_str}\t{species_binary_str}\n")


In [ ]:
hog_stats2 = pd.read_csv('./hogMsaStats2.tsv', sep='\t')
list_XY = [(i,len(hog_stats2[hog_stats2['species_count'] >= i])) for i in range(0,17) ]

plt.plot(*zip(*list_XY))
plt.xlabel('Number of species min cutoff (inclusive)')
plt.ylabel('Number of HOGs')
plt.title('Number of HOGs with at least X species')
plt.ylim(0, 50000)

In [ ]:
speciesStats = pd.read_csv('hogMsaStats2.tsv', delimiter='\t')['species_count']

x_min = 0
x_max = 16
filtered_data = speciesStats[(speciesStats >= x_min) & (speciesStats <= x_max)]

bins = x_max - x_min  # Number of bins equals the range

plt.hist(filtered_data, bins=bins, range=(x_min, x_max))
plt.title('Histogram of species count in HOG')
plt.xlabel('Species count')
plt.ylabel('Frequency')
plt.xlim(x_min, x_max)
plt.grid()
plt.show()

In [ ]:
geneStats = pd.read_csv('hogMsaStats2.tsv', delimiter='\t')['gene_count']

x_min = 0
x_max = 50
filtered_data = geneStats[(geneStats >= x_min) & (geneStats <= x_max)]

bins = x_max - x_min  # Number of bins equals the range

plt.hist(filtered_data, bins=bins, range=(x_min, x_max))
plt.title('Histogram of gene count in HOG')
plt.xlabel('Gene count')
plt.ylabel('Frequency')
plt.xlim(x_min, x_max)
plt.grid()
plt.show()

In [ ]:
hog_Msa_stats = pd.read_csv('hogMsaStats2.tsv', delimiter='\t')

# make a 2d heat map of frequency of BOP vs PACMAD species reporesented in HOGs
# BOP: Osa	Ogl	Ola	Lpe	Hvu	Ata	Tae	Bdi	
# PACMAD: Svi	Pvi	Pha	Msi	Sbi	Zma	Ecu	Pau

pacmad = ['Svi', 'Pvi', 'Pha', 'Msi', 'Sbi', 'Zma', 'Ecu', 'Pau']
bop = ['Osa', 'Ogl', 'Ola', 'Lpe', 'Hvu', 'Ata', 'Tae', 'Bdi']

hog_Msa_stats['BOP'] = hog_Msa_stats[bop].sum(axis=1)
hog_Msa_stats['PACMAD'] = hog_Msa_stats[pacmad].sum(axis=1)

frequency_table = (
    hog_Msa_stats.groupby(['BOP', 'PACMAD'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=range(9), columns=range(9), fill_value=0)
    .sort_index(axis=0, ascending=False)
    .sort_index(axis=1)
)

sns.heatmap(frequency_table, annot=True, fmt="d", cbar_kws={'label': 'Frequency'}, cmap='viridis')
plt.title('Heatmap of BOP vs PACMAD Species Represented in HOGs')
plt.xlabel('PACMAD Count')
plt.ylabel('BOP Count')
plt.tight_layout()
plt.show()

nohup bash run_all.sh &> all.log &


<!-- 
conda activate orthofinder
nohup bash maffter.sh > bashLogs/maffter.log &

nohup bash hmmbuilder.sh > bashLogs/grassHmm.log &
cat ../grassfamHmmLib/* > ../grassHmmDatabase/grassfam
hmmpress ../grassHmmDatabase/grassfam

nohup bash confusionMatrixMaker.sh > bashLogs/confusion.log & -->
